<a href="https://colab.research.google.com/github/7235SYXD/Real-Estate/blob/main/DSP_on_Real_Estate_(Notebook_1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — INSTALL LIBRARIES                                      ║
# ╚══════════════════════════════════════════════════════════════════╝

"""
Packages installed:
    kagglehub       : As the size of the datasets are too large to upload we are using
                      kagglehub link to directly upload the CSV to avoid the upload limit.
    catboost        : Yandex CatBoost are used as the primary base learner for gradient
                      boosted trees.
    shap            : SHapley Additive is used for model interpretability.
    optuna          : Bayesian hyperparameter optimization with Tree-structured
                      Parzen Estimator is used for all the 4 model tunning.
    vaderSentiment  : VADER sentiment analyser is used to extract compound, positive,
                      negative scores per listing.
    yake            : Yet Another Keyword Extractor used to extract the top 3 key
                      phrases per listing description.
    textstat        : Flesch Reading and Flesch-Kincaid grade are calculated and is
                      used to measure listing description readability.
"""

import subprocess, sys
for lib in ["kagglehub","catboost","shap","optuna",
            "vaderSentiment","yake","textstat"]:
    subprocess.run([sys.executable,"-m","pip","install",lib,"-q"])
print("All libraries installed.")


All libraries installed.


In [2]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — GOOGLE DRIVE                                           ║
# ╚══════════════════════════════════════════════════════════════════╝
"""
Why Google Drive:
    Colab sessions are ephemeral — All the content files and in-memory data
    is lost when the session ends or the runtime restarts. Saving it to
    google drive helps us to make the outputs persistent across all the sessions
    and are used to share among the 4 notebooks.

    Mounts Google Drive and creates the shared output directory.

Sets:
    SAVE_DIR (str) : absolute path to the shared Drive directory.
"""

import os, shutil
from google.colab import drive, files

drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/RealEstate_TXNY"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}")

Mounted at /content/drive
Save directory: /content/drive/MyDrive/RealEstate_TXNY


In [3]:
# ╔════════════════════════════════════════════════════╗
# ║  CELL 3 — IMPORT ALL LIBRARIES                     ║
# ╚════════════════════════════════════════════════════╝

"""
Imports all libraries and sets global constants.

Global constants set in this cell:
    SEED (42) : This random seed is used for all operations which helps to ensure
                reproducibility. This random seed is applied to numpy, tensorflow,
                train_test_split, all sklearn models, optuna samplers, KFold splits.

Key imports by category:
    Data wrangling  : numpy, pandas, gc (garbage collection for RAM management)
    Visualisation   : matplotlib.pyplot, seaborn, gridspec
    Preprocessing   : StandardScaler, IterativeImputer (MICE), LabelEncoder
    Models          : RandomForestRegressor, CatBoostRegressor, keras (MLP),
                      HistGradientBoostingRegressor
    Metrics         : r2_score, mean_squared_error, mean_absolute_error,
                      roc_auc_score, f1_score, classification_report
    Tuning          : optuna (TPE sampler, MedianPruner)
    Interpretability: shap (TreeExplainer, permutation_importance)
    NLP             : SentimentIntensityAnalyzer (VADER), yake, textstat

"""

import os
import re
import warnings
from pathlib import Path
import gc # Import the garbage collection module

import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.gridspec as gridspec
import seaborn as sns
import matplotlib.ticker as mticker

# Sklearn
from sklearn.model_selection import train_test_split, KFold, cross_val_predict, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    roc_auc_score, f1_score, classification_report,
    accuracy_score,
)

# Gradient boosting
!pip install catboost
from catboost import CatBoostRegressor, CatBoostClassifier
import xgboost as xgb

# Deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Hyperparameter tuning
!pip install optuna
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Interpretability
import shap

# NLP
!pip install vaderSentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
!pip install yake
import yake
!pip install textstat
import textstat

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.4f}".format)

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("=" * 60)
print("All libraries imported successfully.")
print(f"  pandas     : {pd.__version__}")
print(f"  numpy      : {np.__version__}")
print(f"  tensorflow : {tf.__version__}")
print(f"  sklearn    : OK")
print("=" * 60)

All libraries imported successfully.
  pandas     : 2.2.2
  numpy      : 2.0.2
  tensorflow : 2.20.0
  sklearn    : OK


In [4]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — PII COLUMN REGISTRY + STATE MAP                        ║
# ╚══════════════════════════════════════════════════════════════════╝
"""
Defines PII column lists and state normalisation.

PII lists defined:
    PII_COLS_SAKIB (list)     : 3 columns to drop from SAKIB before processing.
                                brokered_by (agent/broker name), street (property address),
                                prev_sold_date (can identify owner transaction history)
    PII_COLS_POLARTECH (list) : 9 columns to drop from POLARTECH.
                                property_url, property_id, broker_id, agent_name,
                                agency_name, address, street_name, apartment, agent_phone
    PII_COLS_NLP (list)       : 6 columns to drop from TX/NY NLP datasets.
                                 address, street, agent

Functions defined:
    drop_pii_columns(df, pii_list, dataset_name) -> pd.DataFrame
        Drops PII columns that exist in the DataFrame.
        Reports dropped columns and safely ignores missing ones.
        Args:
            df           : input DataFrame
            pii_list     : list of column names to attempt to drop
            dataset_name : string label for print output
        Returns:
            DataFrame with PII columns removed.

    normalise_state -> pd.Series
        Converts state names to standard 2-letter abbreviations.
        Handles: full names ("Texas" → "TX"), already-abbreviated ("TX" → "TX"),
                 lowercase ("florida" → "FL"), mixed case ("New York" → "NY").
        Args:
            series : pd.Series of raw state strings
        Returns:
            pd.Series of 2-letter state abbreviations.

STATE_MAP (dict):
    Maps 50 US state full names + DC to 2-letter USPS abbreviations.

Validation:
    Tests normalise_state() on ["Texas", "New York", "CA", "florida"].
    Expected output: ["TX", "NY", "CA", "FL"].
"""

# PII columns per dataset (lowercase column names after normalisation)
PII_COLS_SAKIB = [
    "brokered_by",     # agent / broker name — PII
    "street",          # full street address — PII
    "prev_sold_date",  # transaction history — quasi-PII
]

PII_COLS_POLARTECH = [
    "property_url",    # URL can identify listing owner — PII
    "property_id",     # internal ID — quasi-PII
    "broker_id",       # broker identifier — PII
    "agent_name",      # personal name — PII
    "agency_name",     # business name — quasi-PII
    "address",         # full street address — PII
    "street_name",     # partial address — PII
    "apartment",       # unit number — PII
    "agent_phone",     # agent phone number - PII
]

PII_COLS_NLP = [
    "address",         # full address — PII
    "street",          # street name — PII
    "agent",           # agent name — PII
    "listing_agent",   # agent name — PII
    "mls_id",          # MLS listing ID — quasi-PII
    "listing_id",      # listing identifier — quasi-PII
]

def drop_pii_columns(df, pii_list, dataset_name):
    """
    Drop PII columns from a DataFrame.
    """
    found = [c for c in pii_list if c in df.columns]
    not_found = [c for c in pii_list if c not in df.columns]
    if found:
        df = df.drop(columns=found)
        print(f"  {dataset_name} — PII dropped: {found}")
    if not_found:
        print(f"  {dataset_name} — PII not present (safe): {not_found}")
    return df

# ── STATE MAP ────────────────────────────────────────────────────────────────
STATE_MAP = {
    "ALABAMA":"AL","ALASKA":"AK","ARIZONA":"AZ","ARKANSAS":"AR",
    "CALIFORNIA":"CA","COLORADO":"CO","CONNECTICUT":"CT","DELAWARE":"DE",
    "FLORIDA":"FL","GEORGIA":"GA","HAWAII":"HI","IDAHO":"ID",
    "ILLINOIS":"IL","INDIANA":"IN","IOWA":"IA","KANSAS":"KS",
    "KENTUCKY":"KY","LOUISIANA":"LA","MAINE":"ME","MARYLAND":"MD",
    "MASSACHUSETTS":"MA","MICHIGAN":"MI","MINNESOTA":"MN","MISSISSIPPI":"MS",
    "MISSOURI":"MO","MONTANA":"MT","NEBRASKA":"NE","NEVADA":"NV",
    "NEW HAMPSHIRE":"NH","NEW JERSEY":"NJ","NEW MEXICO":"NM","NEW YORK":"NY",
    "NORTH CAROLINA":"NC","NORTH DAKOTA":"ND","OHIO":"OH","OKLAHOMA":"OK",
    "OREGON":"OR","PENNSYLVANIA":"PA","RHODE ISLAND":"RI","SOUTH CAROLINA":"SC",
    "SOUTH DAKOTA":"SD","TENNESSEE":"TN","TEXAS":"TX","UTAH":"UT",
    "VERMONT":"VT","VIRGINIA":"VA","WASHINGTON":"WA","WEST VIRGINIA":"WV",
    "WISCONSIN":"WI","WYOMING":"WY","DISTRICT OF COLUMBIA":"DC",
}
for abbr in ["AL","AK","AZ","AR","CA","CO","CT","DE","FL","GA","HI","ID",
             "IL","IN","IA","KS","KY","LA","ME","MD","MA","MI","MN","MS",
             "MO","MT","NE","NV","NH","NJ","NM","NY","NC","ND","OH","OK",
             "OR","PA","RI","SC","SD","TN","TX","UT","VT","VA","WA","WV",
             "WI","WY","DC"]:
    STATE_MAP[abbr] = abbr

def normalise_state(series):
    """Convert any state format to 2-letter abbreviation."""
    return (series.astype(str).str.strip().str.upper()
            .map(lambda x: STATE_MAP.get(x, x)))

# Verify STATE_MAP
test_states = pd.Series(["Texas", "New York", "CA", "florida"])
for raw, norm in zip(test_states, normalise_state(test_states)):
    print(f"  '{raw}' → '{norm}'")
print("State normalisation")
print("PII column registry is defined ")


  'Texas' → 'TX'
  'New York' → 'NY'
  'CA' → 'CA'
  'florida' → 'FL'
State normalisation
PII column registry is defined 


In [5]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — DOWNLOAD ALL 4 DATASETS                                ║
# ╚══════════════════════════════════════════════════════════════════╝
"""
Download all the 4 datasets via kagglehub link.

Design decision — TX+NY only corpus:
    If we use the full 50 state national corpus which contains ~2.7M rows, but
    we have only 2-5% listing descriptions. Due to which the NLP features will
    receive near-identical values for 95% of rows, making the ablation study
    show zero. So i have restricted it to two state which gives 100% NLP
    coverage and enable meaningful ablation comparision.

Variables set:
    path_sakib     (str) : local cache path for SAKIB dataset
    path_polartech (str) : local cache path for POLARTECH dataset
    path_texas     (str) : local cache path for TEXAS 2026 NLP dataset
    path_newyork   (str) : local cache path for NEW YORK 2026 NLP dataset
"""

print("Downloading 4 datasets ...")
path_sakib    = kagglehub.dataset_download("ahmedshahriarsakib/usa-real-estate-dataset")
path_polartech= kagglehub.dataset_download("polartech/500000-us-homes-data-for-sale-properties")
path_texas    = kagglehub.dataset_download("jahnavikachhia23/texas-residential-real-estate-intelligence-2026")
path_newyork  = kagglehub.dataset_download("kanchana1990/new-york-real-estate-data-2026")
print(f"  SAKIB     : {path_sakib}")
print(f"  POLARTECH : {path_polartech}")
print(f"  TEXAS     : {path_texas}")
print(f"  NEW YORK  : {path_newyork}")

Using Colab cache for faster access to the 'usa-real-estate-dataset' dataset.


100%|██████████| 34.6M/34.6M [00:00<00:00, 50.9MB/s]

Extracting files...


100%|██████████| 3.46M/3.46M [00:00<00:00, 96.3MB/s]

Extracting files...


100%|██████████| 3.11M/3.11M [00:00<00:00, 130MB/s]

Extracting files...
  SAKIB     : /kaggle/input/usa-real-estate-dataset
  POLARTECH : /root/.cache/kagglehub/datasets/polartech/500000-us-homes-data-for-sale-properties/versions/1
  TEXAS     : /root/.cache/kagglehub/datasets/jahnavikachhia23/texas-residential-real-estate-intelligence-2026/versions/1
  NEW YORK  : /root/.cache/kagglehub/datasets/kanchana1990/new-york-real-estate-data-2026/versions/1


In [6]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — LOAD CSV FILES                                         ║
# ╚══════════════════════════════════════════════════════════════════╝
"""
Reads the primary CSV from each kagglehub download directory.

Function defined:
    find_csv(folder) -> str
        Walks a directory tree recursively and returns the path of the
        first .csv file found. Handles nested subdirectory structures
        created by kagglehub.
        Args:
            folder (str) : root directory to search
        Returns:
            Absolute path to first .csv file found.
        Raises:
            FileNotFoundError if no .csv is found.

DataFrames loaded:
    df_sakib     : 2,226,382 rows × 10 cols (national, no descriptions)
    df_polartech : ~600,000  rows × 28 cols (national, no descriptions)
    df_texas     : ~12,137   rows × 13 cols (TX only, has descriptions)
    df_newyork   : ~8,273    rows × 11 cols (NY only, has descriptions)

Constants set:
    TEXAS_DESC   (str) : column name for description in Texas dataset
    NEWYORK_DESC (str) : column name for description in New York dataset

Output:
    Prints shape and column list for each dataset for manual verification.
"""

def find_csv(folder):
    for root, dirs, fs in os.walk(folder):
        for f in fs:
            if f.endswith(".csv"):
                return os.path.join(root, f)
    raise FileNotFoundError(f"No CSV in: {folder}")

df_sakib     = pd.read_csv(find_csv(path_sakib),     low_memory=False)
df_polartech = pd.read_csv(find_csv(path_polartech), low_memory=False)
df_texas     = pd.read_csv(find_csv(path_texas),     low_memory=False)
df_newyork   = pd.read_csv(find_csv(path_newyork),   low_memory=False)

for name, df in [("SAKIB",df_sakib),("POLARTECH",df_polartech),
                  ("TEXAS",df_texas),("NEW YORK",df_newyork)]:
    print(f"{name:10s}: {df.shape[0]:,} rows x {df.shape[1]} cols")
    print(f"  Columns: {list(df.columns)}")

TEXAS_DESC   = "description"
NEWYORK_DESC = "description"

SAKIB     : 2,226,382 rows x 12 cols
  Columns: ['brokered_by', 'status', 'price', 'bed', 'bath', 'acre_lot', 'street', 'city', 'state', 'zip_code', 'house_size', 'prev_sold_date']
POLARTECH : 600,000 rows x 28 cols
  Columns: ['property_url', 'property_id', 'address', 'street_name', 'apartment', 'city', 'state', 'latitude', 'longitude', 'postcode', 'price', 'bedroom_number', 'bathroom_number', 'price_per_unit', 'living_space', 'land_space', 'land_space_unit', 'broker_id', 'property_type', 'property_status', 'year_build', 'total_num_units', 'listing_age', 'RunDate', 'agency_name', 'agent_name', 'agent_phone', 'is_owned_by_zillow']
TEXAS     : 12,137 rows x 13 cols
  Columns: ['type', 'sub_type', 'text', 'listPrice', 'sqft', 'stories', 'beds', 'baths', 'baths_full', 'baths_full_calc', 'garage', 'year_built', 'Price_Per_SqFt']
NEW YORK  : 8,273 rows x 11 cols
  Columns: ['type', 'sub_type', 'text', 'listPrice', 'sqft', 'stories', 'beds', 'baths', 'baths_full', 'baths_full_calc', 'garage'